In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_validation import label_forecasts, summarize_forecasts


# 11 Static fOU Convergence Calibration
Label realized first passages directly from observed spread paths, independently of option exits.


In [ ]:
cfg = ResearchConfig()


In [ ]:
forecasts = pd.read_parquet("forecasts.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
trades = pd.read_parquet("trades.parquet")
calibration = label_forecasts(forecasts, test_prices)
calibration.to_parquet("forecast_calibration.parquet")
summary = summarize_forecasts(calibration)
pd.to_pickle(summary, "calibration_summary.pkl")
display(pd.Series(summary))


In [ ]:
traded_forecasts = forecasts.loc[forecasts.forecast_id.isin(trades.forecast_id)] if len(trades) else forecasts.iloc[:0]
traded_calibration = label_forecasts(traded_forecasts, test_prices)
traded_calibration.to_parquet("traded_forecast_calibration.parquet")
traded_summary = summarize_forecasts(traded_calibration)
pd.to_pickle(traded_summary, "traded_forecast_calibration_summary.pkl")
display(pd.Series(traded_summary))


In [ ]:
complete = calibration.loc[calibration.complete_horizon_observed].copy()
if len(complete):
    complete["horizon_bin"] = pd.cut(
        complete.convergence_horizon_trading_days, bins=[0, 5, 20, 63, 126, np.inf]
    )
    buckets = complete.groupby("horizon_bin", observed=True).agg(
        n_forecasts=("pair", "size"),
        mean_prediction=("probability_at_selected_horizon", "mean"),
        observed_rate=("realized_within_selected_horizon", "mean"),
    )
    buckets.index = buckets.index.astype(str)
    buckets.to_parquet("calibration_horizon_buckets.parquet")
    display(buckets)
    buckets[["mean_prediction", "observed_rate"]].plot.bar(figsize=(10, 4))
    plt.tight_layout()
    plt.show()

if len(trades):
    display(trades.groupby("exit_reason").agg(
        n_trades=("pnl", "size"),
        total_pnl=("pnl", "sum"),
        mean_return=("trade_return", "mean"),
    ))
